# Generate Report

Generate a report similar to the paper "On the Use of Agentic Coding" using fix-only PRs from `results/` (generated by `classify_fix_prs.py`).

**Produces:**
- `results/report_figures/` — PNG figures
- `results/report.txt` — text summary with tables

In [6]:
pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.


## Imports & Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as sp_stats

%matplotlib inline

## Paths

In [8]:
DATA_DIR = Path("data_dec2024_feb2026")
OUT_DIR = Path("results")
FIG_DIR = OUT_DIR / "report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUT_DIR / "report.txt"

RESULTS_DIR = Path("results")

## Load Data (Fix PRs Only)

In [9]:
HF_BASE = "https://huggingface.co/datasets/mabujadallah/GitHub-Agentic-PR-Dataset/resolve/main"

print("Loading fix-only data from Hugging Face ...")
all_prs   = pd.read_parquet(f"{HF_BASE}/fix_prs_only.parquet")
agent_prs = all_prs[all_prs["source"] == "agent"].copy()
human_prs = all_prs[all_prs["source"] == "human"].copy()
commits   = pd.read_parquet(f"{HF_BASE}/fix_pr_commits.parquet")
details   = pd.read_parquet(f"{HF_BASE}/fix_pr_commit_details.parquet",
                            columns=["sha", "pr_id", "filename", "additions", "deletions"])

print(f"  Fix Agent PRs : {len(agent_prs):,}")
print(f"  Fix Human PRs : {len(human_prs):,}")
print(f"  Commits       : {len(commits):,}")
print(f"  Details       : {len(details):,}")

Loading fix-only data from Hugging Face ...
  Fix Agent PRs : 48,624
  Fix Human PRs : 300,786
  Commits       : 913,847
  Details       : 3,559,124


## Helper Functions

In [10]:
def word_count(text) -> int:
    if not text or not isinstance(text, str):
        return 0
    return len(text.split())


report_lines: list[str] = []

def section(title: str):
    report_lines.append("")
    report_lines.append("=" * 70)
    report_lines.append(f"  {title}")
    report_lines.append("=" * 70)
    print(f"\n{'=' * 70}\n  {title}\n{'=' * 70}")

def out(line: str = ""):
    report_lines.append(line)
    print(line)

## Dataset Overview

In [11]:
section("Dataset Overview (Fix PRs Only)")
out(f"Date range         : {all_prs['created_at'].min()[:10]} to {all_prs['created_at'].max()[:10]}")
out(f"Total fix PRs      : {len(all_prs):,}")
out(f"  Agent fix PRs    : {len(agent_prs):,}")
out(f"  Human fix PRs    : {len(human_prs):,}")
out(f"Repositories       : {all_prs['repo_name'].nunique():,}")
out(f"Total commits      : {len(commits):,}")
out(f"Unique committers  : {commits['author'].nunique():,}")

if "agent" in agent_prs.columns:
    out("\nAgent fix PR breakdown:")
    for agent, cnt in agent_prs["agent"].value_counts().items():
        out(f"  {agent:20s} {cnt:>8,}")


  Dataset Overview (Fix PRs Only)
Date range         : 2024-12-01 to 2026-02-28
Total fix PRs      : 349,410
  Agent fix PRs    : 48,624
  Human fix PRs    : 300,786
Repositories       : 1,616
Total commits      : 913,847
Unique committers  : 31,494

Agent fix PR breakdown:
  Cursor                 20,516
  Claude_Code            16,990
  Copilot                 8,760
  Devin                   2,358


## RQ1 — Fix PR Change Size

**How do Agent fix PRs differ from Human fix PRs in change size?**

In [12]:
section("RQ1: How do Agent fix PRs differ from Human fix PRs in change size?")

# --- Change size metrics from commit details ---
# Aggregate per PR: files changed, lines added, lines deleted
pr_stats = details.groupby("pr_id").agg(
    files_changed=("filename", "nunique"),
    lines_added=("additions", "sum"),
    lines_deleted=("deletions", "sum"),
).reset_index()

agent_ids = set(agent_prs["id"])
human_ids = set(human_prs["id"])

apr_stats = pr_stats[pr_stats["pr_id"].isin(agent_ids)]
hpr_stats = pr_stats[pr_stats["pr_id"].isin(human_ids)]

out("\nChange size (median):")
for metric in ["files_changed", "lines_added", "lines_deleted"]:
    a_med = apr_stats[metric].median() if len(apr_stats) else 0
    h_med = hpr_stats[metric].median() if len(hpr_stats) else 0
    out(f"  {metric:<20s}  APR: {a_med:>8.0f}   HPR: {h_med:>8.0f}")

# --- Description length ---
agent_prs["desc_words"] = agent_prs["body"].apply(word_count)
human_prs["desc_words"] = human_prs["body"].apply(word_count)
out(f"\nPR description length (median words):  APR: {agent_prs['desc_words'].median():.0f}   HPR: {human_prs['desc_words'].median():.0f}")


  RQ1: How do Agent fix PRs differ from Human fix PRs in change size?

Change size (median):
  files_changed         APR:        2   HPR:        2
  lines_added           APR:       28   HPR:       20
  lines_deleted         APR:       10   HPR:        7

PR description length (median words):  APR: 142   HPR: 85


In [13]:
# Violin plots for change size (matching paper Fig 4 style)
# KDE is computed on log-transformed data for smooth, curvy violins,
# then the axis is relabeled back to original scale.
def log_violin_plot(ax, apr_vals, hpr_vals, ylabel):
    """Plot smooth violins by running KDE in log-space."""
    frames = []
    if len(apr_vals):
        tmp = pd.DataFrame({"val": np.log10(apr_vals.clip(lower=1).values), "Group": "APR"})
        frames.append(tmp)
    if len(hpr_vals):
        tmp = pd.DataFrame({"val": np.log10(hpr_vals.clip(lower=1).values), "Group": "HPR"})
        frames.append(tmp)
    if not frames:
        return
    plot_df = pd.concat(frames, ignore_index=True)
    sns.violinplot(
        data=plot_df, x="Group", y="val", hue="Group",
        palette={"APR": "#5b9bd5", "HPR": "#ed7d31"},
        inner="box", cut=0, density_norm="width", ax=ax, legend=False,
        bw_adjust=0.5,
    )
    # Relabel y-axis ticks back to original scale
    yticks = ax.get_yticks()
    ax.set_yticks(yticks)
    ax.set_yticklabels([f"$10^{{{int(t)}}}$" if t == int(t) else "" for t in yticks])
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ["files_changed", "lines_added", "lines_deleted"]
titles_m = ["Changed Files Count", "Added Lines", "Deleted Lines"]
for ax, metric, title in zip(axes, metrics, titles_m):
    log_violin_plot(ax, apr_stats[metric], hpr_stats[metric], title)
fig.suptitle("Distribution of Change Metrics \u2014 Fix PRs (Agent vs Human)", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "rq1_change_size.png", dpi=150, bbox_inches="tight")
plt.show()
out(f"  -> Figure saved: {FIG_DIR / 'rq1_change_size.png'}")

  -> Figure saved: results\report_figures\rq1_change_size.png


C:\Users\Mahmoudabujadallah\AppData\Local\Temp\ipykernel_32216\3644372083.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## RQ2 — Acceptance / Rejection Rates

**To what extent are Agent fix PRs rejected?**

In [ ]:
section("RQ2: To what extent are Agent fix PRs rejected?")

def merge_rate(df: pd.DataFrame) -> tuple[int, int, float]:
    merged = df["merged_at"].notna().sum()
    total = len(df)
    return merged, total, merged / total * 100 if total else 0

apr_m, apr_t, apr_r = merge_rate(agent_prs)
hpr_m, hpr_t, hpr_r = merge_rate(human_prs)

out(f"Agent PRs  : {apr_m:,} / {apr_t:,} merged  = {apr_r:.1f}%")
out(f"Human PRs  : {hpr_m:,} / {hpr_t:,} merged  = {hpr_r:.1f}%")

# Chi-squared test on merge rates
contingency = np.array([[apr_m, apr_t - apr_m], [hpr_m, hpr_t - hpr_m]])
if contingency.min() > 0:
    chi2, p_chi, _, _ = sp_stats.chi2_contingency(contingency)
    out(f"Chi-squared test: chi2={chi2:.2f}, p={p_chi:.4f}")
    out(f"  {'Statistically significant' if p_chi < 0.05 else 'Not statistically significant'} at alpha=0.05")

# Merge rate by agent tool
if "agent" in agent_prs.columns:
    out("\nMerge rate by agent tool:")
    for agent_name, grp in agent_prs.groupby("agent"):
        m, t, r = merge_rate(grp)
        out(f"  {agent_name:20s}  {m:>6,} / {t:>6,}  = {r:.1f}%")

In [ ]:
# Merge rate bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Agent PRs", "Human PRs"], [apr_r, hpr_r], color=["#5b9bd5", "#ed7d31"])
ax.set_ylabel("Merge Rate (%)")
ax.set_title("Fix PR Acceptance Rate")
ax.set_ylim(0, 100)
for bar, val in zip(bars, [apr_r, hpr_r]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{val:.1f}%",
            ha="center", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "rq2_merge_rate.png", dpi=150)
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'rq2_merge_rate.png'}")

In [ ]:
# --- Time to merge ---
for label, df in [("Agent", agent_prs), ("Human", human_prs)]:
    merged_df = df[df["merged_at"].notna()].copy()
    if len(merged_df) == 0:
        continue
    merged_df["created_dt"] = pd.to_datetime(merged_df["created_at"], utc=True)
    merged_df["merged_dt"] = pd.to_datetime(merged_df["merged_at"], utc=True)
    merged_df["hours_to_merge"] = (merged_df["merged_dt"] - merged_df["created_dt"]).dt.total_seconds() / 3600
    median_h = merged_df["hours_to_merge"].median()
    out(f"\n{label} PRs \u2014 median time to merge: {median_h:.2f} hours")

In [ ]:
# Merge rate by agent (grouped bar)
if "agent" in agent_prs.columns:
    agent_groups = agent_prs.groupby("agent")
    agents_list = sorted(agent_groups.groups.keys())
    merge_rates = []
    for a in agents_list:
        grp = agent_groups.get_group(a)
        _, _, r = merge_rate(grp)
        merge_rates.append(r)

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = sns.color_palette("tab10", len(agents_list))
    ax.bar(agents_list, merge_rates, color=colors)
    ax.set_ylabel("Merge Rate (%)")
    ax.set_title("Fix PR Merge Rate by Agent Tool")
    ax.set_ylim(0, 100)
    for i, (a, r) in enumerate(zip(agents_list, merge_rates)):
        ax.text(i, r + 1, f"{r:.1f}%", ha="center", fontsize=9)
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "rq2_merge_rate_by_agent.png", dpi=150)
    plt.show()
    out(f"  -> Figure saved: {FIG_DIR / 'rq2_merge_rate_by_agent.png'}")

## RQ3 — Revisions

**What proportion of fix PRs are accepted without revisions?**

In [ ]:
section("RQ3: What proportion of fix PRs are accepted without revisions?")

# Count commits per PR
commits_per_pr = commits.groupby("pr_id").size().reset_index(name="num_commits")

apr_commits = commits_per_pr[commits_per_pr["pr_id"].isin(agent_ids)]
hpr_commits = commits_per_pr[commits_per_pr["pr_id"].isin(human_ids)]

# Merged PRs only
merged_agent_ids = set(agent_prs[agent_prs["merged_at"].notna()]["id"])
merged_human_ids = set(human_prs[human_prs["merged_at"].notna()]["id"])

apr_merged_commits = apr_commits[apr_commits["pr_id"].isin(merged_agent_ids)]
hpr_merged_commits = hpr_commits[hpr_commits["pr_id"].isin(merged_human_ids)]

apr_single = (apr_merged_commits["num_commits"] == 1).sum()
hpr_single = (hpr_merged_commits["num_commits"] == 1).sum()
apr_total_merged = len(apr_merged_commits)
hpr_total_merged = len(hpr_merged_commits)

apr_single_pct = apr_single / apr_total_merged * 100 if apr_total_merged else 0
hpr_single_pct = hpr_single / hpr_total_merged * 100 if hpr_total_merged else 0

out(f"Merged without revision (single commit):")
out(f"  Agent PRs : {apr_single:,} / {apr_total_merged:,} = {apr_single_pct:.1f}%")
out(f"  Human PRs : {hpr_single:,} / {hpr_total_merged:,} = {hpr_single_pct:.1f}%")

# Revision commit count distribution (for PRs with >1 commit)
apr_revised = apr_merged_commits[apr_merged_commits["num_commits"] > 1]
hpr_revised = hpr_merged_commits[hpr_merged_commits["num_commits"] > 1]

out(f"\nRevised PRs (>1 commit):")
out(f"  Agent PRs : {len(apr_revised):,}  median commits: {apr_revised['num_commits'].median():.0f}" if len(apr_revised) else "  Agent PRs : 0")
out(f"  Human PRs : {len(hpr_revised):,}  median commits: {hpr_revised['num_commits'].median():.0f}" if len(hpr_revised) else "  Human PRs : 0")

# Mann-Whitney U test on revision counts
if len(apr_revised) > 0 and len(hpr_revised) > 0:
    u_stat, p_val = sp_stats.mannwhitneyu(
        apr_revised["num_commits"], hpr_revised["num_commits"], alternative="two-sided"
    )
    out(f"  Mann-Whitney U: U={u_stat:.0f}, p={p_val:.4f}")
    out(f"  {'Statistically significant' if p_val < 0.05 else 'Not statistically significant'} at alpha=0.05")

In [ ]:
# Revision count violin plot (matching paper Fig 5 style)
fig, ax = plt.subplots(figsize=(6, 4))
frames_rev = []
if len(apr_revised):
    tmp = apr_revised[["num_commits"]].copy()
    tmp["Group"] = "APR"
    frames_rev.append(tmp)
if len(hpr_revised):
    tmp = hpr_revised[["num_commits"]].copy()
    tmp["Group"] = "HPR"
    frames_rev.append(tmp)
if frames_rev:
    plot_df = pd.concat(frames_rev, ignore_index=True)
    # Clip outliers for cleaner visualization
    y_max = min(plot_df["num_commits"].max(), 50)
    plot_df["num_commits"] = plot_df["num_commits"].clip(upper=y_max)
    sns.violinplot(
        data=plot_df, x="Group", y="num_commits", hue="Group",
        palette={"APR": "#5b9bd5", "HPR": "#ed7d31"},
        inner="box", cut=0, density_norm="width", ax=ax, legend=False,
        bw_adjust=0.5,
    )
    ax.set_ylim(0, y_max + 1)
ax.set_ylabel("Revisions Count")
ax.set_xlabel("")
ax.set_title("Revision Commits in Merged Fix PRs (>1 commit)")
fig.tight_layout()
fig.savefig(FIG_DIR / "rq3_revision_commits.png", dpi=150)
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'rq3_revision_commits.png'}")

In [ ]:
# Revision change size (files, lines) for revised PRs
# For revised PRs, compute stats from commits after the first one
def revision_stats(revised_pr_ids: set, commits_df: pd.DataFrame, details_df: pd.DataFrame) -> pd.DataFrame:
    """Get change stats from revision commits (all commits after the first)."""
    # Find first commit per PR (by order in commits table)
    pr_commits_ordered = commits_df[commits_df["pr_id"].isin(revised_pr_ids)].copy()
    first_commits = pr_commits_ordered.groupby("pr_id").first().reset_index()
    first_shas = set(first_commits["sha"])

    # Revision commits = all non-first commits
    rev_shas = set(pr_commits_ordered["sha"]) - first_shas
    rev_details = details_df[details_df["sha"].isin(rev_shas)]

    return rev_details.groupby("pr_id").agg(
        rev_files_changed=("filename", "nunique"),
        rev_lines_added=("additions", "sum"),
        rev_lines_deleted=("deletions", "sum"),
    ).reset_index()

apr_rev_ids = set(apr_revised["pr_id"])
hpr_rev_ids = set(hpr_revised["pr_id"])

if apr_rev_ids or hpr_rev_ids:
    apr_rev_stats = revision_stats(apr_rev_ids, commits, details) if apr_rev_ids else pd.DataFrame()
    hpr_rev_stats = revision_stats(hpr_rev_ids, commits, details) if hpr_rev_ids else pd.DataFrame()

    out("\nRevision change size (median, revised PRs only):")
    for metric in ["rev_files_changed", "rev_lines_added", "rev_lines_deleted"]:
        a_med = apr_rev_stats[metric].median() if len(apr_rev_stats) else 0
        h_med = hpr_rev_stats[metric].median() if len(hpr_rev_stats) else 0
        out(f"  {metric:<25s}  APR: {a_med:>8.0f}   HPR: {h_med:>8.0f}")

In [ ]:
# Violin plots for revision change size (log-space KDE for smooth curves)
if apr_rev_ids or hpr_rev_ids:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    rev_metrics = ["rev_files_changed", "rev_lines_added", "rev_lines_deleted"]
    rev_titles = ["Changed Files (Revisions)", "Added Lines (Revisions)", "Deleted Lines (Revisions)"]
    for ax, metric, title in zip(axes, rev_metrics, rev_titles):
        apr_vals = apr_rev_stats[metric].dropna() if len(apr_rev_stats) else pd.Series(dtype=float)
        hpr_vals = hpr_rev_stats[metric].dropna() if len(hpr_rev_stats) else pd.Series(dtype=float)
        log_violin_plot(ax, apr_vals, hpr_vals, title)
    fig.suptitle("Revision Change Size \u2014 Fix PRs (Agent vs Human)", y=1.02, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "rq3_revision_change_size.png", dpi=150, bbox_inches="tight")
    plt.show()
    out(f"  -> Figure saved: {FIG_DIR / 'rq3_revision_change_size.png'}")

## Additional Statistics

In [ ]:
section("Additional Statistics")

# PR state breakdown
out("PR state breakdown:")
for label, df in [("Agent", agent_prs), ("Human", human_prs)]:
    out(f"  {label}:")
    for state, cnt in df["state"].value_counts().items():
        out(f"    {state:12s} {cnt:>8,}")

In [ ]:
# Monthly trends
all_prs["created_month"] = pd.to_datetime(all_prs["created_at"], utc=True).dt.to_period("M")
monthly = all_prs.groupby(["created_month", "is_agent"]).size().unstack(fill_value=0)
monthly.columns = ["Human", "Agent"] if False in monthly.columns else monthly.columns

out("\nMonthly PR volume:")
out(f"{'Month':<12} {'Agent':>8} {'Human':>8} {'Total':>8}")
out("-" * 40)
for period in monthly.index:
    row = monthly.loc[period]
    agent_val = row.get(True, row.get("Agent", 0))
    human_val = row.get(False, row.get("Human", 0))
    total = agent_val + human_val
    out(f"{str(period):<12} {int(agent_val):>8,} {int(human_val):>8,} {int(total):>8,}")

In [ ]:
# Monthly trend chart
fig, ax = plt.subplots(figsize=(10, 5))
months_str = [str(p) for p in monthly.index]
agent_vals = [monthly.loc[p].get(True, monthly.loc[p].get("Agent", 0)) for p in monthly.index]
human_vals = [monthly.loc[p].get(False, monthly.loc[p].get("Human", 0)) for p in monthly.index]
ax.plot(months_str, agent_vals, "o-", label="Agent PRs", color="#5b9bd5")
ax.plot(months_str, human_vals, "s-", label="Human PRs", color="#ed7d31")
ax.set_xlabel("Month")
ax.set_ylabel("Number of PRs")
ax.set_title("Monthly Fix PR Volume: Agent vs Human")
ax.legend()
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "monthly_trend.png", dpi=150)
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'monthly_trend.png'}")

In [ ]:
# Top repositories
out("\nTop 10 repos by total PRs:")
top_repos = all_prs["repo_name"].value_counts().head(10)
for repo, cnt in top_repos.items():
    agent_cnt = len(all_prs[(all_prs["repo_name"] == repo) & (all_prs["is_agent"] == True)])
    out(f"  {repo:50s}  total: {cnt:>5,}  agent: {agent_cnt:>5,}")

## Save Report

In [ ]:
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print(f"\nReport saved to {REPORT_PATH}")
print(f"Figures saved to {FIG_DIR}/")
print("Done!")